In [1]:
import os
import sys
from pathlib import Path

# Locate the repo root as the directory containing this notebook.
# __vsc_ipynb_file__ is injected by VS Code; fall back to cwd for other environments.
try:
    NOTEBOOK_DIR = str(Path(__vsc_ipynb_file__).parent.resolve())
except NameError:
    NOTEBOOK_DIR = str(Path.cwd())

os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

print(f"Working directory: {os.getcwd()}")

Working directory: /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF


In [2]:
import pandas as pd
from pyspedas import CDAWeb
from l1_pipeline import get_one_day_swmf_input, create_position_file
from l1_combine import create_combined_l1_files

cda = CDAWeb()

This unreleased version of SpacePy is not supported by the SpacePy team.


07-Mar-26 12:39:58: /opt/miniconda3/lib/python3.13/site-packages/spacepy/time.py:2448: UserWarning: Leapseconds may be out of date. Use spacepy.toolbox.update(leapsecs=True)
  _read_leaps()



In [3]:
# -----------------------------------------------------------------------
# Date range to process.
# Uncomment the rolling-window block to always run the last N days.
# -----------------------------------------------------------------------

# today = pd.Timestamp.utcnow().strftime('%Y-%m-%d')
# days = pd.date_range(end=today, periods=7).strftime('%Y-%m-%d').tolist()

from plot_l1_may2024 import plot_day

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

start_time = pd.Timestamp.now()

# Seed the window: download the day before day-1 (context only) and day-1.
day_before = (pd.Timestamp(days[0]) -
              pd.Timedelta(days=1)).strftime('%Y-%m-%d')
get_one_day_swmf_input(day_before, cda)
get_one_day_swmf_input(days[0], cda)
create_position_file(days[0], cda)

# Crawling window: download tomorrow, combine today (yesterday + tomorrow
# as context), plot, then advance.  Each day is downloaded exactly once.
for i, day in enumerate(days):
    prev_day = day_before if i == 0 else days[i - 1]

    if i < len(days) - 1:
        next_day = days[i + 1]
        get_one_day_swmf_input(next_day, cda)
        create_position_file(next_day, cda)
    else:
        next_day = None

    create_combined_l1_files(day, prev_day=prev_day, next_day=next_day)
    try:
        plot_day(day)
    except Exception as exc:
        print(f"  Plot error for {day}: {exc}")
    print(f"Completed: {day}")

end_time = pd.Timestamp.now()
print(f"\nDone. Elapsed: {end_time - start_time}")

Querying CDAWeb...


07-Mar-26 12:40:06: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240430_v07.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240430_v07.cdf


07-Mar-26 12:40:06: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240430_v07.cdf complete, 0.235 MB in 0.4 sec (0.659 MB/sec) (transfer_normal)
07-Mar-26 12:40:06: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf
07-Mar-26 12:40:06: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf complete, 0.238 MB in 0.4 sec (0.620 MB/sec) (transfer_normal)
07-Mar-26 12:40:07: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240430_v11.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propa


Processing ace...
Saved L1/2024/04/30/L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240430000000_e20240430235959_p20240501021914_pub.nc.gz -> cdf_temp/dscovr_f1m_20240430.nc.gz
  Downloaded oe_m1m_dscovr_s20240430000000_e20240430235959_p20240501021331_pub.nc.gz -> cdf_temp/dscovr_m1m_20240430.nc.gz
Saved L1/2024/04/30/L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
Saved L1/2024/04/30/L1_wind.dat
Cleaning up wind CDFs...
Querying CDAWeb...


07-Mar-26 12:40:22: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240501_v07.cdf


...

07-Mar-26 12:55:59: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240531_v07.cdf complete, 0.244 MB in 0.3 sec (0.729 MB/sec) (transfer_normal)
07-Mar-26 12:55:59: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240601_v07.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240601_v07.cdf
07-Mar-26 12:55:59: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240601_v07.cdf complete, 0.242 MB in 0.4 sec (0.642 MB/sec) (transfer_normal)
07-Mar-26 12:55:59: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/swepam/level_2_cdaweb/swe_h0/2024/ac_h0_swe_20240531_v11.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propa


Processing ace...
Saved L1/2024/05/31/L1_ace.dat
Cleaning up ace CDFs...

Processing DSCOVR (NGDC)...
  Downloaded oe_f1m_dscovr_s20240531000000_e20240531235959_p20240601022216_pub.nc.gz -> cdf_temp/dscovr_f1m_20240531.nc.gz
  Downloaded oe_m1m_dscovr_s20240531000000_e20240531235959_p20240601021645_pub.nc.gz -> cdf_temp/dscovr_m1m_20240531.nc.gz
Saved L1/2024/05/31/L1_dscovr.dat
  Cleaning up DSCOVR NGDC files...

Processing wind...
Rotating WIND Plasma data from GSE to GSM...
Saved L1/2024/05/31/L1_wind.dat
Cleaning up wind CDFs...

--- Generating L1_satpos.dat for 2024-05-31 ---
Querying Position Data (11:00-13:00 UT)...


07-Mar-26 12:56:12: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240531_v07.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240531_v07.cdf


07-Mar-26 12:56:13: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/ace/mag/level_2_cdaweb/mfi_h0/2024/ac_h0_mfi_20240531_v07.cdf complete, 0.244 MB in 0.3 sec (0.712 MB/sec) (transfer_normal)
07-Mar-26 12:56:13: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/wind/mfi/mfi_h0/2024/wi_h0_mfi_20240531_v05.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/wind/mfi/mfi_h0/2024/wi_h0_mfi_20240531_v05.cdf
07-Mar-26 12:56:13: Download of /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/wind/mfi/mfi_h0/2024/wi_h0_mfi_20240531_v05.cdf complete, 2.497 MB in 0.5 sec (4.973 MB/sec) (transfer_normal)
07-Mar-26 12:56:13: Downloading https://cdaweb.gsfc.nasa.gov/sp_phys/data/dscovr/orbit/pre_or/2024/dscovr_orbit_pre_20240531_v04.cdf to /Users/connordimarco/Library/CloudStorage/OneDrive-Umich/Work/Propagation/SWMF-IMF/cdf_temp/dscovr/orbit/pre_or/202

Saved L1/2024/05/31/L1_satpos.dat

Processing L1 data for 2024-05-30  (context window: ['2024-05-29', '2024-05-30', '2024-05-31'])...
  Running plasma quality assessment for all satellites...
  ACE quality: flagged bad: {'Ux': 0, 'Uy': 0, 'Uz': 0, 'rho': 1085, 'T': 329}
  DSCOVR quality: flagged bad: {'Ux': 406, 'Uy': 269, 'Uz': 1428, 'rho': 1489, 'T': 3014}
  WIND quality: flagged bad: {'Ux': 283, 'Uy': 283, 'Uz': 283, 'rho': 283, 'T': 1990}
  -> Running Despike (Growth & Change limits)...
Created L1/2024/05/30/L1_combined.dat
  -> Propagating to 14 Re (89194 km)...
    Created L1/2024/05/30/IMF_14Re.dat
  -> Propagating to 32 Re (203872 km)...
    Created L1/2024/05/30/IMF_32Re.dat
Saved plots/2024_05_30.png
Completed: 2024-05-30

Processing L1 data for 2024-05-31  (context window: ['2024-05-30', '2024-05-31'])...
  Running plasma quality assessment for all satellites...
  ACE quality: flagged bad: {'Ux': 0, 'Uy': 0, 'Uz': 0, 'rho': 500, 'T': 1}
  DSCOVR quality: flagged bad: {'Ux': 

In [4]:
# -----------------------------------------------------------------------
# Context plots: each day with ±6 h of the neighbouring days.
# Vertical dashed lines mark midnight boundaries so you can immediately
# spot any jumps between days.  Shaded regions are the context windows.
# Output goes to  plots/with_context/
# Standalone — can be run without running cell 3 first.
# -----------------------------------------------------------------------

import pandas as pd
from plot_l1_may2024 import plot_day_with_context

days = pd.date_range(
    start='2024-05-01', end='2024-05-31'
).strftime('%Y-%m-%d').tolist()

context_start = pd.Timestamp.now()

for day in days:
    try:
        plot_day_with_context(day, context_hours=6,
                              output_dir='plots/with_context')
    except Exception as exc:
        print(f"  Context-plot error for {day}: {exc}")

context_end = pd.Timestamp.now()
print(f"\nContext plots done. Elapsed: {context_end - context_start}")

Saved plots/with_context/2024_05_01.png
Saved plots/with_context/2024_05_02.png
Saved plots/with_context/2024_05_03.png
Saved plots/with_context/2024_05_04.png
Saved plots/with_context/2024_05_05.png
Saved plots/with_context/2024_05_06.png
Saved plots/with_context/2024_05_07.png
Saved plots/with_context/2024_05_08.png
Saved plots/with_context/2024_05_09.png
Saved plots/with_context/2024_05_10.png
Saved plots/with_context/2024_05_11.png
Saved plots/with_context/2024_05_12.png
Saved plots/with_context/2024_05_13.png
Saved plots/with_context/2024_05_14.png
Saved plots/with_context/2024_05_15.png
Saved plots/with_context/2024_05_16.png
Saved plots/with_context/2024_05_17.png
Saved plots/with_context/2024_05_18.png
Saved plots/with_context/2024_05_19.png
Saved plots/with_context/2024_05_20.png
Saved plots/with_context/2024_05_21.png
Saved plots/with_context/2024_05_22.png
Saved plots/with_context/2024_05_23.png
Saved plots/with_context/2024_05_24.png
Saved plots/with_context/2024_05_25.png
